In [1]:
!pip install torch torchvision --quiet
!pip install einops --quiet

In [2]:
import sys
import os
import glob
import cv2
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    jaccard_score,
    classification_report
)

from pathlib import Path

import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange


In [3]:
dataset_dir = Path('../EWS-Dataset')

train_dir = dataset_dir / 'train'
test_dir = dataset_dir / 'test'
validation_dir = dataset_dir / 'validation'

In [ ]:
PIXELS_PER_IMAGE = 5000

FEATURE_MODES = {
    "RGB": ["r", "g", "b"],
    "RGB+HSV": ["r", "g", "b", "h", "s", "v"],
    "RGB+HSV+Texture": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var"],
    "RGB+HSV+Texture+VegIdx": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var", "exg", "ndi", "veg", "exg_blur3", "exg_blur7"],
}

In [3]:
class WheatDataset(Dataset):
    def __init__(self, img_dir, transform=None, mask_transform=None):
        self.img_dir = Path(img_dir)
        self.images = sorted([
            p for p in self.img_dir.glob('*.png')
            if '_mask' not in p.stem
        ])
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.images)
    
    def __getItem__(self, idx):
        img_path = self.img[idx]
        mask_path = self.img_dir / (img_path.stem + '_mask.png')

        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convery('L')

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        mask = (mask < 0.5).float()
        return image, mask
        
img_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = WheatDataset('../EWS-Dataset/train',      img_transform, mask_transform)
val_dataset   = WheatDataset('../EWS-Dataset/validation', img_transform, mask_transform)
test_dataset  = WheatDataset('../EWS-Dataset/test',       img_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)
test_loader  = DataLoader(test_dataset,  batch_size=8)

In [ ]:
import torch.nn.functional as F

def dice_loss(pred, target, smooth=1):
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(logits, masks):
    logits = F.interpolate(logits, size=masks.shape[-2:],
                           mode = 'bilinear', align_corners=False)
    ce = F.cross_entropy(logits, masks)
    pred = logits.softmax(dim=1)[:,1]
    dl = dice_loss(pred, masks.float())
    return ce + dl

In [6]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

optimizer = AdamW(model.parameters(), lr=6e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=50)


NameError: name 'model' is not defined